# 07 Results summary (preliminary)
Models trained on raw vs trimmed 2019 audio. Re-run as runs finish; 2021 sets appear once evaluated.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from audio_df.helpers.research.runs import collect, eer_table

MODELS = ["a1", "a4", "a2", "d1", "d2", "d3"]
ROUND2 = ["a1", "a4", "a2", "d1", "d2 L12", "d3 L12"]  # D2/D3 with WavLM layer 12
ALL = ROUND2 + ["d2 L6", "d3 L6"]                     # round 3: WavLM layer 6
NAMES = {"a1": "A1 LFCC-GMM", "a4": "A4 LFCC-MLP", "a2": "A2 WavLM", "d1": "D1 concat",
         "d2 L12": "D2 attn, layer 12", "d3 L12": "D3 attn, layer 12",
         "d2 L6": "D2 attn, layer 6", "d3 L6": "D3 attn, layer 6"}
TESTS = ["eval", "eval_mp3_32k", "eval_noise_10db", "eval_opus_16k", "eval_resample_8k", "itw",
         "asvspoof2021_eval", "asvspoof2021_hidden", "asvspoof2021df_eval", "asvspoof2021df_hidden"]
# A1/A4 were scored on 10,000-clip stratified DF samples; every other model on the full DF sets.
SETS = {"dev": "2019 dev", "eval": "2019 eval", "eval_clean10k": "2019 eval 10k", "eval_mp3_32k": "MP3 32k",
        "eval_noise_10db": "Noise 10 dB", "eval_opus_16k": "Opus 16k", "eval_resample_8k": "Resample 8k",
        "itw": "In-the-Wild", "asvspoof2021_eval": "2021 LA eval", "asvspoof2021_hidden": "2021 LA hidden",
        "asvspoof2021df_eval": "2021 DF eval", "asvspoof2021df_hidden": "2021 DF hidden"}
RAW, TRIM = "#86b6ef", "#1c5cab"  # blue 250 / 550: one hue, validated as an ordered pair
INK, MUTED, RULE = "#0b0b0b", "#52514e", "#d9d8d4"
BLUES = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": RULE,
                     "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

table = eer_table(collect("../runs", MODELS))
table = table[[s for s in SETS if s in table]]
SETS["mean"] = "mean (10 test sets)"  # only for models scored on all 10 (raw-trained runs lack 2021)
table["mean"] = table[[s for s in TESTS if s in table]].mean(axis=1, skipna=False)
table.rename(columns=SETS).round(2)

## Raw-trained vs trimmed-trained, per test set
EER %, lower is better. Light dot = trained on raw audio, dark dot = trained on trimmed.

In [ ]:
tests = [s for s in ["eval", "itw", "eval_noise_10db", "eval_mp3_32k", "asvspoof2021_eval", "asvspoof2021_hidden"]
         if s in table]
fig, axes = plt.subplots(1, len(tests), figsize=(3.2 * len(tests), 3.6), sharey=True)
y = np.arange(len(ROUND2))[::-1]
for ax, t in zip(axes, tests):
    raw = [table[t].get((m, "raw"), np.nan) for m in ROUND2]
    trim = [table[t].get((m, "trimmed"), np.nan) for m in ROUND2]
    ax.hlines(y, raw, trim, color=RULE, lw=2, zorder=1)
    ax.scatter(raw, y, s=70, color=RAW, edgecolor="#fcfcfb", lw=2, zorder=2, label="raw-trained")
    ax.scatter(trim, y, s=70, color=TRIM, edgecolor="#fcfcfb", lw=2, zorder=3, label="trimmed-trained")
    for yi, v in zip(y, trim):
        if not np.isnan(v):
            ax.annotate(f"{v:.1f}", (v, yi), xytext=(0, 8), textcoords="offset points", ha="center",
                        fontsize=8, color=MUTED)
    ax.set_title(SETS[t], fontsize=10, loc="left")
    ax.set_xlim(0, max(55, np.nanmax(raw + trim) + 5))
    ax.grid(axis="x", color=RULE, lw=0.6)
    ax.set_axisbelow(True)
    ax.set_xlabel("EER %")
axes[0].set_yticks(y, [NAMES[m] for m in ROUND2])
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=2, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

## Round 2 → round 3: WavLM layer 12 → 6
Trimmed training, EER %. The only change is the layer feeding the attention; the black tick is A2 (WavLM only).

In [ ]:
tests = [s for s in TESTS if s in table]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
y = np.arange(len(tests))[::-1]
for ax, (r2, r3, name) in zip(axes, [("d2 L12", "d2 L6", "D2 one-way attention"),
                                     ("d3 L12", "d3 L6", "D3 two-way attention")]):
    before = [table.loc[(r2, "trimmed"), t] for t in tests]
    after = [table.loc[(r3, "trimmed"), t] for t in tests]
    ax.hlines(y, before, after, color=RULE, lw=2, zorder=1)
    ax.scatter(before, y, s=70, color=RAW, edgecolor="#fcfcfb", lw=2, zorder=2, label="round 2 (layer 12)")
    ax.scatter(after, y, s=70, color=TRIM, edgecolor="#fcfcfb", lw=2, zorder=3, label="round 3 (layer 6)")
    ax.scatter([table.loc[("a2", "trimmed"), t] for t in tests], y, marker="|", s=220, color=INK, lw=2,
               zorder=4, label="A2")
    ax.set_title(name, fontsize=10, loc="left")
    ax.set_xlabel("EER %")
    ax.grid(axis="x", color=RULE, lw=0.6)
    ax.set_axisbelow(True)
axes[0].set_yticks(y, [SETS[t] for t in tests])
fig.legend(*axes[0].get_legend_handles_labels(), loc="upper center", ncol=3, fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

## Trimmed-trained models, every test set
EER %; darker = worse.

In [ ]:
trimmed = table.xs("trimmed", level="train").reindex([m for m in ALL if (m, "trimmed") in table.index])
fig, ax = plt.subplots(figsize=(1.0 * trimmed.shape[1] + 2, 0.5 * len(trimmed) + 1.2))
ax.imshow(trimmed.to_numpy(), cmap=BLUES, vmin=0, vmax=60, aspect="auto")
for (i, j), v in np.ndenumerate(trimmed.to_numpy()):
    ax.text(j, i, "–" if np.isnan(v) else f"{v:.1f}", ha="center", va="center", fontsize=9,
            color="#fcfcfb" if v > 30 else INK)
ax.set_xticks(range(trimmed.shape[1]), [SETS[c] for c in trimmed.columns], rotation=35, ha="right")
ax.set_yticks(range(len(trimmed)), [NAMES[m] for m in trimmed.index])
ax.tick_params(length=0)
for s in ax.spines.values():
    s.set_visible(False)
fig.tight_layout()
plt.show()

## Conclusion
- **Round 1 → 2 (trimmed training):** removing the silence shortcut made the WavLM models much better
  (A2 2019 eval 12.0% → 7.6%). The LFCC-only models did not improve (A1 ≈ 20.5%, A4 ≈ 37%).
- **Round 2 → 3 (WavLM layer 12 → 6, nothing else changed):** both attention models now beat A2 on
  average over the 10 test sets (mean EER D2 11.0%, D3 11.5%, A2 12.5%). The largest gains are on 2021 LA
  eval (14.3% → 10.1%) and MUSAN noise (17.0% → 12.0% for D3).
- **2021 DF (full 533,928 clips):** D2 10.0% vs A2 10.7%, D3 11.2%. D1 falls to 22.6%: concatenated LFCC
  breaks under DF's media codecs, as it did under MP3. A1/A4 DF numbers are on 10,000-clip stratified
  samples (notebook 01), so treat them as estimates.
- **In-the-Wild stays around 20% for every model** (A2 19.2%, D2 19.9%, D3 22.7%). No single WavLM layer
  gets it below 20.7% (notebook 04), so this is a limit of frozen WavLM trained on 2019 LA, not of the
  fusion. Next step: augmentation, fine-tuning WavLM or more varied training data.
- One training run per model: gaps under about 1% (e.g. 2019 eval) may be noise.